這份 Notebook 是[《Deep Learning with Python》第三版](https://www.manning.com/books/deep-learning-with-python-third-edition)的範例。為方便閱讀，只保留可執行的程式碼與小節標題，不包含書中的說明文字、圖片與虛擬碼。

**建議搭配本書閱讀，才能掌握各段程式的用途與原理。**

作者公開英文版也可在 [deeplearningwithpython.io](https://deeplearningwithpython.io) 線上閱讀。

In [0]:
!pip install keras keras-hub --upgrade -q

In [0]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [0]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## TensorFlow、PyTorch、JAX 與 Keras 簡介

### 深度學習框架簡史

### 這些框架之間的關係

### TensorFlow 簡介

#### TensorFlow 入門

##### TensorFlow 中的張量與變數

###### 常數張量

**程式 3.1 全 1 或全 0 的張量**

In [0]:
import tensorflow as tf
tf.ones(shape=(2, 1))

In [0]:
tf.zeros(shape=(2, 1))

In [0]:
tf.constant([1, 2, 3], dtype="float32")

###### 亂數張量

**程式 3.2 亂數張量**

In [0]:
x = tf.random.normal(shape=(3, 1), mean=0., stddev=1.)
print(x)

In [0]:
x = tf.random.uniform(shape=(3, 1), minval=0., maxval=1.)
print(x)

###### 張量賦值與 Variable 類別

**程式 3.3 NumPy 陣列可被賦值**

In [0]:
import numpy as np

x = np.ones(shape=(2, 2))
x[0, 0] = 0.0

**程式 3.5 建立 tf.Variable**

In [0]:
v = tf.Variable(initial_value=tf.random.normal(shape=(3, 1)))
print(v)

**程式 3.6 為 Variable 指派值**

In [0]:
v.assign(tf.ones((3, 1)))

**程式 3.7 為 Variable 的子集指派值**

In [0]:
v[0, 0].assign(3.)

**程式 3.8 使用 assign_add**

In [0]:
v.assign_add(tf.ones((3, 1)))

##### 張量運算：在 TensorFlow 中進行數學運算

**程式 3.9 TensorFlow 中的一些基本數學運算**

In [0]:
a = tf.ones((2, 2))
b = tf.square(a)
c = tf.sqrt(a)
d = b + c
e = tf.matmul(a, b)
f = tf.concat((a, b), axis=0)

In [0]:
def dense(inputs, W, b):
    return tf.nn.relu(tf.matmul(inputs, W) + b)

##### TensorFlow 中的梯度：深入探討 GradientTape API

**程式 3.10 使用 GradientTape**

In [0]:
input_var = tf.Variable(initial_value=3.0)
with tf.GradientTape() as tape:
    result = tf.square(input_var)
gradient = tape.gradient(result, input_var)

**程式 3.11 在 GradientTape 中使用常數張量輸入**

In [0]:
input_const = tf.constant(3.0)
with tf.GradientTape() as tape:
    tape.watch(input_const)
    result = tf.square(input_const)
gradient = tape.gradient(result, input_const)

**程式 3.12 利用巢狀梯度運算紀錄計算二階梯度**

In [0]:
time = tf.Variable(0.0)
with tf.GradientTape() as outer_tape:
    with tf.GradientTape() as inner_tape:
        position = 4.9 * time**2
    speed = inner_tape.gradient(position, time)
acceleration = outer_tape.gradient(speed, time)

##### 透過編譯加速 TensorFlow 函式

In [0]:
@tf.function
def dense(inputs, W, b):
    return tf.nn.relu(tf.matmul(inputs, W) + b)

In [0]:
@tf.function(jit_compile=True)
def dense(inputs, W, b):
    return tf.nn.relu(tf.matmul(inputs, W) + b)

#### 端到端範例：純 TensorFlow 線性分類器

**程式 3.13 在二維平面上生成兩個類別的亂數資料點**

In [0]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

**程式 3.14 將兩個類別堆疊成形狀為 (2000, 2) 的陣列**

In [0]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

**程式 3.15 產生對應的目標值（0 與 1）**

In [0]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

**程式 3.16 繪製兩個點類別**

In [0]:
import matplotlib.pyplot as plt

plt.scatter(inputs[:, 0], inputs[:, 1], c=targets[:, 0])
plt.show()

**程式 3.17 建立線性分類器的變數**

In [0]:
input_dim = 2
output_dim = 1
W = tf.Variable(initial_value=tf.random.uniform(shape=(input_dim, output_dim)))
b = tf.Variable(initial_value=tf.zeros(shape=(output_dim,)))

**程式 3.18 正向傳播函式**

In [0]:
def model(inputs, W, b):
    return tf.matmul(inputs, W) + b

**程式 3.19 均方誤差損失函數**

In [0]:
def mean_squared_error(targets, predictions):
    per_sample_losses = tf.square(targets - predictions)
    return tf.reduce_mean(per_sample_losses)

**程式 3.20 訓練步驟函式**

In [0]:
learning_rate = 0.1

@tf.function(jit_compile=True)
def training_step(inputs, targets, W, b):
    with tf.GradientTape() as tape:
        predictions = model(inputs, W, b)
        loss = mean_squared_error(predictions, targets)
    grad_loss_wrt_W, grad_loss_wrt_b = tape.gradient(loss, [W, b])
    W.assign_sub(grad_loss_wrt_W * learning_rate)
    b.assign_sub(grad_loss_wrt_b * learning_rate)
    return loss

**程式 3.21 批次訓練迴圈**

In [0]:
for step in range(40):
    loss = training_step(inputs, targets, W, b)
    print(f"Loss at step {step}: {loss:.4f}")

In [0]:
predictions = model(inputs, W, b)
plt.scatter(inputs[:, 0], inputs[:, 1], c=predictions[:, 0] > 0.5)
plt.show()

In [0]:
x = np.linspace(-1, 4, 100)
y = -W[0] / W[1] * x + (0.5 - b) / W[1]
plt.plot(x, y, "-r")
plt.scatter(inputs[:, 0], inputs[:, 1], c=predictions[:, 0] > 0.5)

#### TensorFlow 的獨特之處

### PyTorch 簡介

#### PyTorch 入門

##### PyTorch 中的張量與參數

###### 常數張量

**程式 3.22 全 1 或全 0 的張量**

In [0]:
import torch
torch.ones(size=(2, 1))

In [0]:
torch.zeros(size=(2, 1))

In [0]:
torch.tensor([1, 2, 3], dtype=torch.float32)

###### 亂數張量

**程式 3.23 亂數張量**

In [0]:
torch.normal(
mean=torch.zeros(size=(3, 1)),
std=torch.ones(size=(3, 1)))

In [0]:
torch.rand(3, 1)

###### 張量賦值與 Parameter 類別

In [0]:
x = torch.zeros(size=(2, 1))
x[0, 0] = 1.
x

**程式 3.24 建立 PyTorch 參數**

In [0]:
x = torch.zeros(size=(2, 1))
p = torch.nn.parameter.Parameter(data=x)

##### 張量運算：在 PyTorch 中進行數學運算

**程式 3.25 PyTorch 中的一些基本數學運算**

In [0]:
a = torch.ones((2, 2))
b = torch.square(a)
c = torch.sqrt(a)
d = b + c
e = torch.matmul(a, b)
f = torch.cat((a, b), dim=0)

In [0]:
def dense(inputs, W, b):
    return torch.nn.relu(torch.matmul(inputs, W) + b)

##### 在 PyTorch 中計算梯度

**程式 3.26 使用 .backward() 計算梯度**

In [0]:
input_var = torch.tensor(3.0, requires_grad=True)
result = torch.square(input_var)
result.backward()
gradient = input_var.grad
gradient

In [0]:
result = torch.square(input_var)
result.backward()
input_var.grad

In [0]:
input_var.grad = None

#### 端到端範例：純 PyTorch 線性分類器

In [0]:
input_dim = 2
output_dim = 1

W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

In [0]:
def model(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [0]:
def mean_squared_error(targets, predictions):
    per_sample_losses = torch.square(targets - predictions)
    return torch.mean(per_sample_losses)

In [0]:
learning_rate = 0.1

def training_step(inputs, targets, W, b):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    grad_loss_wrt_W, grad_loss_wrt_b = W.grad, b.grad
    with torch.no_grad():
        W -= grad_loss_wrt_W * learning_rate
        b -= grad_loss_wrt_b * learning_rate
    W.grad = None
    b.grad = None
    return loss

##### 使用 Module 類別封裝狀態與運算

**程式 3.27 定義 torch.nn.Module**

In [0]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.zeros(output_dim))

    def forward(self, inputs):
        return torch.matmul(inputs, self.W) + self.b

In [0]:
model = LinearModel()

In [0]:
torch_inputs = torch.tensor(inputs)
output = model(torch_inputs)

In [0]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [0]:
def training_step(inputs, targets):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    optimizer.step()
    model.zero_grad()
    return loss

##### 透過編譯加速 PyTorch 模組

In [0]:
compiled_model = torch.compile(model)

In [0]:
@torch.compile
def dense(inputs, W, b):
    return torch.nn.relu(torch.matmul(inputs, W) + b)

#### PyTorch 的獨特之處

### JAX 簡介

#### JAX 入門

#### JAX 中的張量

**程式 3.28 全 1 或全 0 的張量**

In [0]:
from jax import numpy as jnp
jnp.ones(shape=(2, 1))

In [0]:
jnp.zeros(shape=(2, 1))

In [0]:
jnp.array([1, 2, 3], dtype="float32")

#### JAX 中的亂數生成

**程式 3.29 亂數張量**

In [0]:
np.random.normal(size=(3,))

In [0]:
np.random.normal(size=(3,))

In [0]:
def apply_noise(x, seed):
    np.random.seed(seed)
    x = x * np.random.normal((3,))
    return x

seed = 1337
y = apply_noise(x, seed)
seed += 1
z = apply_noise(x, seed)

In [0]:
import jax

seed_key = jax.random.key(1337)

In [0]:
seed_key = jax.random.key(0)
jax.random.normal(seed_key, shape=(3,))

**程式 3.30 在 JAX 中使用隨機種子**

In [0]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

In [0]:
jax.random.normal(seed_key, shape=(3,))

In [0]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

In [0]:
new_seed_key = jax.random.split(seed_key, num=1)[0]
jax.random.normal(new_seed_key, shape=(3,))

##### 張量賦值

**程式 3.31 修改 JAX 陣列中的數值**

In [0]:
x = jnp.array([1, 2, 3], dtype="float32")
new_x = x.at[0].set(10)

##### 張量運算：在 JAX 中進行數學運算

**程式 3.32 JAX 中的一些基本數學運算**

In [0]:
a = jnp.ones((2, 2))
b = jnp.square(a)
c = jnp.sqrt(a)
d = b + c
e = jnp.matmul(a, b)
e *= d

In [0]:
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

##### 在 JAX 中計算梯度

In [0]:
def compute_loss(input_var):
    return jnp.square(input_var)

In [0]:
grad_fn = jax.grad(compute_loss)

In [0]:
input_var = jnp.array(3.0)
grad_of_loss_wrt_input_var = grad_fn(input_var)

##### JAX 梯度計算最佳實務

###### 同時回傳損失值

In [0]:
grad_fn = jax.value_and_grad(compute_loss)
output, grad_of_loss_wrt_input_var = grad_fn(input_var)

###### 取得複雜函數的梯度

###### 回傳輔助輸出

##### 使用 @jax.jit 加速 JAX 函式

In [0]:
@jax.jit
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

#### 端到端範例：純 JAX 線性分類器

In [0]:
def model(inputs, W, b):
    return jnp.matmul(inputs, W) + b

def mean_squared_error(targets, predictions):
    per_sample_losses = jnp.square(targets - predictions)
    return jnp.mean(per_sample_losses)

In [0]:
def compute_loss(state, inputs, targets):
    W, b = state
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    return loss

In [0]:
grad_fn = jax.value_and_grad(compute_loss)

In [0]:
learning_rate = 0.1

@jax.jit
def training_step(inputs, targets, W, b):
    loss, grads = grad_fn((W, b), inputs, targets)
    grad_wrt_W, grad_wrt_b = grads
    W = W - grad_wrt_W * learning_rate
    b = b - grad_wrt_b * learning_rate
    return loss, W, b

In [0]:
input_dim = 2
output_dim = 1

W = jax.numpy.array(np.random.uniform(size=(input_dim, output_dim)))
b = jax.numpy.array(np.zeros(shape=(output_dim,)))
state = (W, b)
for step in range(40):
    loss, W, b = training_step(inputs, targets, W, b)
    print(f"Loss at step {step}: {loss:.4f}")

#### JAX 的獨特之處

### Keras 簡介

#### Keras 入門

##### 選擇後端框架

In [0]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras

#### 層（layer）：深度學習的建構積木

##### Keras 中的基礎 Layer 類別

**程式 3.33 在 Keras 中從頭實作簡單的密集層**

In [0]:
import keras

class SimpleDense(keras.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = activation

    def build(self, input_shape):
        batch_dim, input_dim = input_shape
        self.W = self.add_weight(
            shape=(input_dim, self.units), initializer="random_normal"
        )
        self.b = self.add_weight(shape=(self.units,), initializer="zeros")

    def call(self, inputs):
        y = keras.ops.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

In [0]:
my_dense = SimpleDense(units=32, activation=keras.ops.relu)
input_tensor = keras.ops.ones(shape=(2, 784))
output_tensor = my_dense(input_tensor)
print(output_tensor.shape)

##### 自動形狀推論：動態建構層

In [0]:
from keras import layers

layer = layers.Dense(32, activation="relu")

In [0]:
from keras import models
from keras import layers

model = models.Sequential(
    [
        layers.Dense(32, activation="relu"),
        layers.Dense(32),
    ]
)

In [0]:
model = keras.Sequential(
    [
        SimpleDense(32, activation="relu"),
        SimpleDense(64, activation="relu"),
        SimpleDense(32, activation="relu"),
        SimpleDense(10, activation="softmax"),
    ]
)

#### 從層到模型

#### 「compile」步驟：設定學習流程

In [0]:
model = keras.Sequential([keras.layers.Dense(1)])
model.compile(
    optimizer="rmsprop",
    loss="mean_squared_error",
    metrics=["accuracy"],
)

In [0]:
model.compile(
    optimizer=keras.optimizers.RMSprop(),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

#### 選擇損失函數

#### 理解 fit 方法

**程式 3.34 透過 NumPy 資料呼叫 fit**

In [0]:
history = model.fit(
    inputs,
    targets,
    epochs=5,
    batch_size=128,
)

In [0]:
history.history

#### 在驗證資料上監控損失與指標

**程式 3.35 使用 validation_data 引數**

In [0]:
model = keras.Sequential([keras.layers.Dense(1)])
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.1),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

indices_permutation = np.random.permutation(len(inputs))
shuffled_inputs = inputs[indices_permutation]
shuffled_targets = targets[indices_permutation]

num_validation_samples = int(0.3 * len(inputs))
val_inputs = shuffled_inputs[:num_validation_samples]
val_targets = shuffled_targets[:num_validation_samples]
training_inputs = shuffled_inputs[num_validation_samples:]
training_targets = shuffled_targets[num_validation_samples:]
model.fit(
    training_inputs,
    training_targets,
    epochs=5,
    batch_size=16,
    validation_data=(val_inputs, val_targets),
)

#### 推論：在訓練成功後使用模型

In [0]:
predictions = model.predict(val_inputs, batch_size=128)
print(predictions[:10])